# Leakage and diagnostic publication analysis
Attach the sealed leakage inputs and the three completed result sets listed in the artifact index. Run on CPU.


In [ ]:
from pathlib import Path, PurePosixPath
from zipfile import ZipFile
import hashlib, json, shutil, stat, subprocess, sys, tempfile, time
from importlib import metadata

STARTED = time.time()
EXPECTED = 'ac776b466b7b7e01432d607adf8f5724c2c60f84d702c279e8fd80df9a760de6'
BUNDLE_SHA = 'cbb54e6b58ef60b3f28dcbec7457e17c44de46a49451ab4209f17c4eec97106b'
SEARCH = Path('/kaggle/input')
WORK = Path('/kaggle/working')
def file_sha(p):
    with Path(p).open('rb') as f: return hashlib.file_digest(f, 'sha256').hexdigest()
stage = Path(tempfile.gettempdir()) / 'msld_revision_inputs' / EXPECTED
marker = stage / 'revision_manifest.json'
if not marker.is_file() or file_sha(marker) != EXPECTED:
    blobs = list(SEARCH.rglob('msld_revision_payload.bin'))
    if not blobs:
        for archive in SEARCH.rglob('MSLD_Revision_Inputs.zip'):
            with ZipFile(archive) as z:
                if 'msld_revision_payload.bin' in z.namelist():
                    if z.getinfo('msld_revision_payload.bin').file_size > 100_000_000: raise ValueError('Unexpected payload size.')
                    target = stage.parent / (EXPECTED + '.bin'); target.parent.mkdir(parents=True, exist_ok=True)
                    with z.open('msld_revision_payload.bin') as a, target.open('wb') as b: shutil.copyfileobj(a, b)
                    blobs.append(target)
    matches = [p for p in blobs if file_sha(p) == BUNDLE_SHA]
    if not matches: raise FileNotFoundError('Attach the private msld-revision-inputs dataset made from MSLD_Revision_Inputs.zip.')
    stage.mkdir(parents=True, exist_ok=True)
    with ZipFile(matches[0]) as z:
        if sum(i.file_size for i in z.infolist()) > 200_000_000: raise ValueError('Unexpected uncompressed input size.')
        for info in z.infolist():
            parts = PurePosixPath(info.filename.replace('\\','/'))
            if parts.is_absolute() or '..' in parts.parts or any(':' in x for x in parts.parts) or stat.S_ISLNK(info.external_attr >> 16): raise ValueError('Invalid archive path.')
            if info.is_dir(): continue
            target = stage.joinpath(*parts.parts); target.parent.mkdir(parents=True,exist_ok=True)
            with z.open(info) as a, target.open('wb') as b: shutil.copyfileobj(a,b)
if file_sha(marker) != EXPECTED: raise ValueError('Revision manifest checksum mismatch.')
PACKAGE = stage
print('Revision inputs found. Manifest', EXPECTED)

In [ ]:
required = {'numpy': '2.1.3', 'pandas': '2.3.3', 'scipy': '1.16.3', 'scikit-learn': '1.6.1'}
changes = []
for package, version in required.items():
    try: installed = metadata.version(package)
    except metadata.PackageNotFoundError: installed = None
    if installed != version: changes.append(f'{package}=={version}')
if changes: subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps', *changes], check=True)
print('Analysis packages ready.')

In [ ]:
cmd = [sys.executable, str(PACKAGE/'code/merge_revision.py'), '--package', str(PACKAGE), '--search', str(SEARCH), '--out', str(WORK/'Revision_Final_Report'), '--manifest', EXPECTED]
subprocess.run(cmd, check=True)